# Bitcoin Trading Bot
Fetches BTC/USDT price & RSI from Binance and sends signals to Discord.

In [ ]:
# Install dependencies
!pip install -q requests pandas ta

In [ ]:
import requests
import pandas as pd
from ta.momentum import RSIIndicator
import time
from getpass import getpass

SYMBOL = "BTCUSDT"

# Enter your Discord webhook URL securely
DISCORD_WEBHOOK = getpass("Paste your Discord Webhook URL: ")

In [ ]:
def safe_request(url):
    try:
        return requests.get(url, timeout=5).json()
    except Exception as e:
        print(f"Request failed: {e}")
        return None

def get_price():
    data = safe_request(f"https://api.binance.com/api/v3/ticker/price?symbol={SYMBOL}")
    if isinstance(data, dict) and "price" in data:
        return float(data["price"])
    return None

def get_klines():
    data = safe_request(f"https://api.binance.com/api/v3/klines?symbol={SYMBOL}&interval=1m&limit=100")
    if not isinstance(data, list) or len(data) == 0:
        print(f"Unexpected klines response: {data}")
        return None
    return [float(k[4]) for k in data]

def get_rsi(closes):
    df = pd.DataFrame(closes, columns=["close"])
    return RSIIndicator(df["close"], window=14).rsi().iloc[-1]

def generate_signal(rsi):
    if rsi < 30:
        return "BUY", 80
    elif rsi > 70:
        return "SELL", 80
    return "HOLD", 50

def send_to_discord(message):
    try:
        resp = requests.post(DISCORD_WEBHOOK, json={"content": message}, timeout=5)
        if resp.status_code not in (200, 204):
            print(f"Discord error: {resp.status_code}")
    except Exception as e:
        print(f"Discord send failed: {e}")

def run_cycle():
    price = get_price()
    closes = get_klines()

    if not price or not closes:
        print("Data fetch failed, skipping cycle")
        return

    rsi = get_rsi(closes)
    signal, confidence = generate_signal(rsi)

    signal_emoji = {"BUY": "🟢 BUY", "SELL": "🔴 SELL", "HOLD": "🟡 HOLD"}[signal]

    msg = f"""🚀 AI Trading Signal

Asset: {SYMBOL}
Price: ${price:,.2f}
RSI: {round(rsi, 2)}

Signal: {signal_emoji}
Confidence: {confidence}%

Time: {time.strftime('%Y-%m-%d %H:%M:%S')} UTC"""

    print(msg)
    send_to_discord(msg)

In [ ]:
# Run once
run_cycle()

In [ ]:
# Run continuously every 60 seconds (interrupt kernel to stop)
INTERVAL_SECONDS = 60

print(f"Starting loop — signal every {INTERVAL_SECONDS}s. Use Runtime > Interrupt to stop.")
while True:
    run_cycle()
    time.sleep(INTERVAL_SECONDS)